In [1]:
import pandas as pd

In [2]:
old_df = pd.read_parquet("data/dataset/ta_nlp_sector.parquet")
master_df = pd.read_parquet("data/dataset/master_df_60rf.parquet")


In [3]:
train_start = pd.Timestamp('2014-01-01')
train_end = pd.Timestamp('2015-08-01')
val_end = pd.Timestamp('2015-10-01')
test_end = pd.Timestamp('2016-01-01')

In [4]:
master_df.columns.to_list()

['date',
 'open',
 'high',
 'low',
 'close',
 'adj_close',
 'volume',
 'ticker',
 'ema_12',
 'ema_26',
 'ema_50',
 'macd_12_26_9',
 'macdh_12_26_9',
 'macds_12_26_9',
 'rsi_14',
 'stochrsik_14_14_3_3',
 'stochrsid_14_14_3_3',
 'atrr_14',
 'bb_upper',
 'bb_middle',
 'bb_lower',
 'obv',
 'ret_1d',
 'roll_ret_1d',
 'roll_ret_5d',
 'roll_ret_20d',
 'sentiment',
 'emotion_anger',
 'emotion_disgust',
 'emotion_fear',
 'emotion_joy',
 'emotion_neutral',
 'emotion_sadness',
 'emotion_surprize',
 'emotion_anger_pct',
 'emotion_disgust_pct',
 'emotion_fear_pct',
 'emotion_joy_pct',
 'emotion_neutral_pct',
 'emotion_sadness_pct',
 'emotion_surprize_pct',
 'positive_emotion',
 'negative_emotion',
 'uncertainty_emotion',
 'positive_emotion_pct',
 'negative_emotion_pct',
 'uncertainty_emotion_pct',
 'stance_label',
 'finbert_label',
 'stance_score',
 'finbert_score',
 'finbert_up',
 'finbert_down',
 'finbert_neutral',
 'sector',
 'company_name',
 'sector_open_mean',
 'sector_high_mean',
 'sector_low

In [5]:
# Calculate train/val/test proportions by rows
df = master_df.copy()


feature_columns = [
    'open', 'high', 'low', 'close', 'volume',
    'roll_ret_1d', 'roll_ret_5d', 
    'roll_ret_20d',
    
    'ema_12', 'ema_26', 'ema_50', 'macd_12_26_9', 'macdh_12_26_9',
    'macds_12_26_9', 'rsi_14', 'stochrsik_14_14_3_3', 'stochrsid_14_14_3_3',
    'atrr_14', 'bb_upper', 'bb_middle', 'bb_lower', 'obv',
]


columns_to_check = [

    # Sentiment (single)
    'sentiment',

    # Emotions
    'emotion_anger', 'emotion_disgust', 'emotion_fear', 'emotion_joy',
    'emotion_neutral', 'emotion_sadness', 'emotion_surprize',
    'emotion_anger_pct', 'emotion_disgust_pct', 'emotion_fear_pct',
    'emotion_joy_pct', 'emotion_neutral_pct', 'emotion_sadness_pct',
    'emotion_surprize_pct',

    # Unified emotion
    'positive_emotion', 'negative_emotion', 'uncertainty_emotion',
    'positive_emotion_pct', 'negative_emotion_pct', 'uncertainty_emotion_pct',

    # Stance
    'stance_label', 'stance_score',

    # FinBERT
    'finbert_label', 'finbert_score', 'finbert_up', 'finbert_down',
    'finbert_neutral',

    # Sector aggregates
    'sector_open_mean', 'sector_high_mean', 'sector_low_mean', 'sector_close_mean',
    'sector_volume_mean', 'sector_ret_1d', 'sector_ret_5d',
    'sector_ret_20d', 'sector_range', 'sector_vol_20d',
    'ema_12_sector', 'ema_26_sector', 'ema_50_sector', 'macd_12_26_9_sector',
    'macdh_12_26_9_sector', 'macds_12_26_9_sector', 'rsi_14_sector',
    'sector_bb_upper', 'sector_bb_middle', 'sector_bb_lower',
    'market_close', 'sector_rel_strength', 'sector_dispersion_1d',

    # Meta regression predictions
    'reg_pred_ret_1d_LSTM', 'reg_pred_ret_1d_BiLSTM', 'reg_pred_ret_1d_GRU',
    'reg_pred_ret_1d_BiGRU',

    # Regression errors/diagnostics
    'reg_abs_err_lag1_LSTM', 'reg_mae_20_LSTM',
    'reg_rmse_20_LSTM', 'reg_dir_acc_20_LSTM',

    'reg_abs_err_lag1_BiLSTM',
    'reg_mae_20_BiLSTM', 'reg_rmse_20_BiLSTM', 'reg_dir_acc_20_BiLSTM',

    'reg_abs_err_lag1_GRU',
    'reg_mae_20_GRU', 'reg_rmse_20_GRU', 'reg_dir_acc_20_GRU',

    'reg_abs_err_lag1_BiGRU',
    'reg_mae_20_BiGRU', 'reg_rmse_20_BiGRU', 'reg_dir_acc_20_BiGRU',

    # Meta classification predictions
    'cls_prob_up_1d_LSTM', 'cls_prob_up_1d_BiLSTM', 'cls_prob_up_1d_GRU',
    'cls_prob_up_1d_BiGRU',

    # Classification diagnostics
    'cls_brier_20_LSTM', 'cls_logloss_20_LSTM', 'cls_acc_20_LSTM',

    'cls_brier_20_BiLSTM', 'cls_logloss_20_BiLSTM', 'cls_acc_20_BiLSTM',

    'cls_brier_20_GRU', 'cls_logloss_20_GRU', 'cls_acc_20_GRU',

    'cls_brier_20_BiGRU', 'cls_logloss_20_BiGRU', 'cls_acc_20_BiGRU',
]

print(f"Initial master_df shape: {master_df.shape}")

df = df.dropna(subset=feature_columns).sort_values(['ticker','date'])
old_df = old_df.dropna(subset=feature_columns).sort_values(['ticker','date'])

df = df.dropna(subset=columns_to_check)

print(f"After dropping NaNs in selected columns, master_df shape: {df.shape}")

df.reset_index(drop=True, inplace=True)

print(df)

df['date'] = pd.to_datetime(df['date']).dt.normalize()

train_mask = (df['date'] >= train_start) & (df['date'] < train_end)
val_mask   = (df['date'] >= train_end) & (df['date'] < val_end)
test_mask  = (df['date'] >= val_end) & (df['date'] < test_end)

n_total = len(df)
n_train = int(train_mask.sum())
n_val   = int(val_mask.sum())
n_test  = int(test_mask.sum())
n_other = n_total - (n_train + n_val + n_test)

def pct(n):
    return (n / n_total * 100.0) if n_total > 0 else 0.0

print(f'Total rows master DF: {n_total}')
print(f'Train: {n_train} ({pct(n_train):.2f}%)')
print(f'Val:   {n_val} ({pct(n_val):.2f}%)')
print(f'Test:  {n_test} ({pct(n_test):.2f}%)')
print(f'Other: {n_other} ({pct(n_other):.2f}%)')



# Calculate train/val/test proportions by rows
old_df['date'] = pd.to_datetime(old_df['date']).dt.normalize()
columns_to_check = [
                    'sentiment',
                    
                    'emotion_anger', 'emotion_disgust', 'emotion_fear', 'emotion_joy',
                    'emotion_neutral', 'emotion_sadness', 'emotion_surprize',
                    'emotion_anger_pct', 'emotion_disgust_pct', 'emotion_fear_pct',
                    'emotion_joy_pct', 'emotion_neutral_pct', 'emotion_sadness_pct',
                    'emotion_surprize_pct', 
                    
                    'positive_emotion', 'negative_emotion','uncertainty_emotion', 
                    'positive_emotion_pct', 'negative_emotion_pct','uncertainty_emotion_pct', 
                    
                    'stance_label', 'stance_score', 
                    
                    'finbert_label', 'finbert_score', 'finbert_up', 'finbert_down',
                    'finbert_neutral', 
                    
                    'sector_open_mean', 'sector_high_mean', 'sector_low_mean', 'sector_close_mean',
                    'sector_volume_mean', 'sector_ret_1d', 'sector_ret_5d',
                    'sector_ret_20d', 'sector_range', 'sector_vol_20d', 
                    
                    'ema_12_sector','ema_26_sector', 'ema_50_sector', 'macd_12_26_9_sector',
                    'macdh_12_26_9_sector', 'macds_12_26_9_sector', 'rsi_14_sector',
                    'sector_bb_upper', 'sector_bb_middle', 'sector_bb_lower',
                    'market_close', 'sector_rel_strength', 'sector_dispersion_1d'
                ]

print(f"Initial old_df shape: {old_df.shape}")

old_df = old_df.dropna(subset=columns_to_check)

print(f"After dropping NaNs in selected columns, old_df shape: {old_df.shape}")

old_df.reset_index(drop=True, inplace=True)

print(old_df)

train_mask = (old_df['date'] >= train_start) & (old_df['date'] < train_end)
val_mask   = (old_df['date'] >= train_end) & (old_df['date'] < val_end)
test_mask  = (old_df['date'] >= val_end) & (old_df['date'] < test_end)

n_total = len(old_df)
n_train = int(train_mask.sum())
n_val   = int(val_mask.sum())
n_test  = int(test_mask.sum())
n_other = n_total - (n_train + n_val + n_test)

def pct(n):
    return (n / n_total * 100.0) if n_total > 0 else 0.0

print(f'Total rows old DF: {n_total}')
print(f'Train: {n_train} ({pct(n_train):.2f}%)')
print(f'Val:   {n_val} ({pct(n_val):.2f}%)')
print(f'Test:  {n_test} ({pct(n_test):.2f}%)')
print(f'Other: {n_other} ({pct(n_other):.2f}%)')


Initial master_df shape: (108592, 118)
After dropping NaNs in selected columns, master_df shape: (81700, 118)
            date       open       high        low      close  adj_close  \
0     2013-11-19  74.147141  74.768570  73.995712  74.221428  69.059044   
1     2013-11-20  74.175713  74.345711  73.475716  73.571426  68.454254   
2     2013-11-21  73.942856  74.458572  73.381432  74.448570  69.270386   
3     2013-11-22  74.217140  74.594284  74.075714  74.257141  69.092285   
4     2013-11-25  74.431427  75.124283  74.428574  74.820000  69.615990   
...          ...        ...        ...        ...        ...        ...   
81695 2017-08-25  76.559998  77.129997  76.430000  76.720001  76.720001   
81696 2017-08-28  76.900002  76.940002  76.260002  76.470001  76.470001   
81697 2017-08-29  76.209999  76.489998  76.080002  76.449997  76.449997   
81698 2017-08-30  76.239998  76.449997  76.059998  76.099998  76.099998   
81699 2017-08-31  76.269997  76.489998  76.050003  76.330002  76.